# Product LCI data prep

Reads the **product LCI** tab of the RAW LCA Google Sheet - the formulas that translate a product spec (beam span, wall length, membrane coverage) into material/activity amounts - and writes `data/processed/product_lci.csv` for `03_productComparison.ipynb`.

Also reads `data/data_sources/baseline_eol_assumptions.csv` (hand-maintained, see the note in that file) and writes `data/processed/baseline_eol.csv`.

Every formula is evaluated at functional unit = 1 and compared with the sheet's own `activity_amount` column, so a broken/renamed formula fails loudly here instead of in the model.

In [1]:
import re
import sys
import pandas as pd
sys.path.insert(0, ".")
from lca_engine import evaluate_formula, formula_rhs, formula_variables

SHEET_ID = "1Vx1XDlohZulOEaFFgliiqx3JaZp4rxqQ-LC8B0_qGW4"
GID_PRODUCT_LCI = "1891550642"
OUTPUT_DIR = "../../data/processed"
SOURCES_DIR = "../../data/data_sources"


In [2]:
url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID_PRODUCT_LCI}"
raw = pd.read_csv(url, na_values=["n/a"])
raw = raw[raw["case_name"].notna()].copy()
print(f"{len(raw)} product-LCI rows, {raw['case_name'].nunique()} cases")
raw.head()

14 product-LCI rows, 10 cases


,comparison_raw_timber,comparison_raw_cfw,comparison_raw_biopol,comparison_raw_knit,element_type,case_type,case_name,functional_unit_amount,functional_unit,activity_name,activity_amount,activity_unit,formula_functionalUnitAmount_to_activityAmount,comments
0,True,False,False,False,beam,conventional bio-based,glulam beam,1.0,m (beam span),glulam timber,0.00083,m3,m3 = (span / 20) * ((span / 20) / 3) * span,volume (m3) = depth (span / 20) * width (depth...
1,True,True,False,False,beam,conventional fossil-based,reinforced concrete beam,1.0,m (beam span),concrete,7.83333,kg,kg = span * (span / 12) * (0.5 * (span / 12)) ...,kg = span * depth (span / 12) * width (0.5 * d...
2,True,True,False,False,beam,conventional fossil-based,reinforced concrete beam,1.0,m (beam span),steel,0.50000,kg,kg = (span * (span / 12) * (0.5 * (span / 12))...,kg = span * depth (span / 12) * width (0.5 * d...
3,True,True,False,False,beam,conventional fossil-based,reinforced concrete beam,1.0,m (beam span),formwork,0.22500,kg,kg = span * (2 * (span/12) + (0.5*span/12)) * ...,kg plywood formwork = span * (2 * depth (span ...
4,False,False,False,False,beam,conventional bio-based,"""traditional"" coreless filament winded beam",1.0,m (beam span),CFW beam,550.00000,m,m = span * 550,check with Fabian and Otto Stuttgart scientifi...


In [3]:
comparison_cols = [c for c in raw.columns if c.startswith("comparison_raw_")]
rows = []
for _, r in raw.iterrows():
    rhs = formula_rhs(r["formula_functionalUnitAmount_to_activityAmount"])
    vars_ = formula_variables(rhs)
    if len(vars_) != 1:
        raise ValueError(f"{r['case_name']} / {r['activity_name']}: expected exactly one spec variable, got {vars_}")
    fu = r["functional_unit"]                       # e.g. 'm (beam span)'
    m = re.match(r"\s*(\S+)\s*\((.*)\)", fu)
    rows.append({
        **{c: ("n/a" if pd.isna(r[c]) else str(r[c]).upper()) for c in comparison_cols},
        "element_type": r["element_type"], "case_type": r["case_type"], "case_name": r["case_name"],
        "spec_var": vars_[0], "spec_unit": m.group(1) if m else fu, "spec_label": m.group(2) if m else fu,
        "activity_name": r["activity_name"], "activity_unit": r["activity_unit"],
        "formula": rhs, "sheet_amount_at_fu1": r["activity_amount"], "comments": r["comments"],
    })
df = pd.DataFrame(rows)

# a case must use one spec variable across all of its activities
bad = df.groupby("case_name")["spec_var"].nunique()
assert (bad == 1).all(), f"cases with >1 spec variable: {bad[bad > 1].index.tolist()}"
df

,comparison_raw_timber,comparison_raw_cfw,comparison_raw_biopol,comparison_raw_knit,element_type,case_type,case_name,spec_var,spec_unit,spec_label,activity_name,activity_unit,formula,sheet_amount_at_fu1,comments
0,TRUE,FALSE,FALSE,FALSE,beam,conventional bio-based,glulam beam,span,m,beam span,glulam timber,m3,(span / 20) * ((span / 20) / 3) * span,0.00083,volume (m3) = depth (span / 20) * width (depth...
1,TRUE,TRUE,FALSE,FALSE,beam,conventional fossil-based,reinforced concrete beam,span,m,beam span,concrete,kg,span * (span / 12) * (0.5 * (span / 12)) * 240...,7.83333,kg = span * depth (span / 12) * width (0.5 * d...
2,TRUE,TRUE,FALSE,FALSE,beam,conventional fossil-based,reinforced concrete beam,span,m,beam span,steel,kg,(span * (span / 12) * (0.5 * (span / 12)) * 24...,0.50000,kg = span * depth (span / 12) * width (0.5 * d...
3,TRUE,TRUE,FALSE,FALSE,beam,conventional fossil-based,reinforced concrete beam,span,m,beam span,formwork,kg,span * (2 * (span/12) + (0.5*span/12)) * 0.018...,0.22500,kg plywood formwork = span * (2 * depth (span ...
4,FALSE,FALSE,FALSE,FALSE,beam,conventional bio-based,"""traditional"" coreless filament winded beam",span,m,beam span,CFW beam,m,span * 550,550.00000,check with Fabian and Otto Stuttgart scientifi...
5,FALSE,FALSE,TRUE,FALSE,wall,conventional bio-based,adobe brick for interior wall,length,m,wall length,adobe brick,kg,length * 3 * (3/10) * 1700,1530.00000,kg = wall length * height (3m) * thickness (he...
6,FALSE,FALSE,TRUE,FALSE,wall,conventional fossil-based,clay brick for interior wall,length,m,wall length,clay brick,kg,length * 3 * 0.115 * 0.84 * 1900,550.62000,kg = wall length * height (3m) * thickness (0....
7,FALSE,FALSE,TRUE,FALSE,wall,conventional fossil-based,clay brick for interior wall,length,m,wall length,lime mortar,kg,length * 3 * 0.115 * 0.16 * 1900,104.88000,kg = wall volume * 16% * density (1900 kg/m3)
8,FALSE,FALSE,FALSE,TRUE,membrane,conventional fossil-based,synthetic polymer architectural membrane,coverage,m2,membrane coverage,polyester fiber,kg,coverage * 0.3,0.30000,kg = coverage (m2) * density (0.3 kg/m2)
9,FALSE,FALSE,FALSE,TRUE,membrane,conventional fossil-based,synthetic polymer architectural membrane,coverage,m2,membrane coverage,weaving,kg,coverage * 0.3,0.30000,kg = coverage (m2) * density (0.3 kg/m2)


In [4]:
# ── Test: every formula at FU = 1 must reproduce the sheet's activity_amount ──
fu_amount = raw.set_index(raw.index)["functional_unit_amount"].values
errors = []
for (_, r), fu in zip(df.iterrows(), fu_amount):
    val = evaluate_formula(r["formula"], {r["spec_var"]: fu})
    if abs(val - r["sheet_amount_at_fu1"]) > 1e-3 * max(1, abs(r["sheet_amount_at_fu1"])) + 1e-5:
        errors.append((r["case_name"], r["activity_name"], val, r["sheet_amount_at_fu1"]))
print("OK - all formulas reproduce the sheet values at FU = 1" if not errors else errors)
assert not errors

OK - all formulas reproduce the sheet values at FU = 1


In [5]:
df.to_csv(f"{OUTPUT_DIR}/product_lci.csv", index=False)
print(f"Wrote {OUTPUT_DIR}/product_lci.csv ({len(df)} rows)")
print(df.groupby(["element_type", "case_name"]).agg(spec_var=("spec_var", "first"), n_activities=("activity_name", "size")))

Wrote ../../data/processed/product_lci.csv (14 rows)
                                                          spec_var  \
element_type case_name                                               
beam         "traditional" coreless filament winded beam      span   
             glulam beam                                      span   
             raw_cfw                                          span   
             raw_timber                                       span   
             reinforced concrete beam                         span   
membrane     raw_knit                                     coverage   
             synthetic polymer architectural membrane     coverage   
wall         adobe brick for interior wall                  length   
             clay brick for interior wall                   length   
             raw_biopol                                     length   

                                                          n_activities  
element_type case_name           

## Baseline end-of-life assumptions

The sheet gives no EoL for conventional products, and the existing waste-treatment burdens (`waste composting/recycling/incineration/landfilling`) are biowaste/wood datasets that are wrong for concrete, steel or brick. `baseline_eol_assumptions.csv` therefore lists, per baseline activity, a density (`kg_per_unit`) and the routes with shares and the unit-burden names used for the treatment (`treatment_burden_name`) and optional avoided-burden credit (`credit_burden_name`). **These shares are assumptions to be confirmed.**

## Machine parameters (small vs. large scale)

The `processes_typicalValues` tab lists typical machine weight, lifetime, output rate and power per machining process, for small and large scale. `03_productComparison.ipynb` uses the large-scale values for the 'scaled up' variant of each RAW case (see `scale_up_case` in `cases.py`). The tab has no gid in the CSV-export URL pattern used above, so the whole spreadsheet is downloaded as xlsx and the tab is read by name.

In [6]:
import io, urllib.request
xlsx = urllib.request.urlopen(f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=xlsx").read()
mp = pd.read_excel(io.BytesIO(xlsx), sheet_name="processes_typicalValues")
mp.columns = ["machining_process", "parameter", "unit", "small_scale", "large_scale"]
mp = mp.dropna(subset=["machining_process", "parameter"])
mp.to_csv(f"{OUTPUT_DIR}/machine_params.csv", index=False)
print(f"Wrote {OUTPUT_DIR}/machine_params.csv ({len(mp)} rows, processes: {mp['machining_process'].unique().tolist()})")
mp.pivot(index="machining_process", columns="parameter", values="large_scale")

Wrote ../../data/processed/machine_params.csv (16 rows, processes: ['3D printing', 'CNC knitting', 'CNC milling', 'Coreless filament winding'])


parameter,machine lifetime,machine weight,output efficiency,power
machining_process,,,,
3D printing,40000.0,1200.0,15.0,2.0
CNC knitting,50000.0,2000.0,1.5,4.0
CNC milling,70000.0,5000.0,10.0,20.0
Coreless filament winding,50000.0,5000.0,10.0,25.0


## RAW case definitions

The four `raw_*` tabs of the sheet hold everything about the RAW cases except the product mass (which comes from `product_LCIs`): BOMs, process chains, repair settings, end-of-life shares. They are exported to `data/processed/raw_*.csv` and turned into cases by `load_raw_cases` in `cases.py`, which contains no case parameters itself. The checks below stop the export if a tab is inconsistent.

In [7]:
RAW_TABS = ["raw_cases", "raw_bom", "raw_chain", "raw_parameters"]
raw_tabs = {t: pd.read_excel(io.BytesIO(xlsx), sheet_name=t) for t in RAW_TABS}
for t, d in raw_tabs.items():
    d.dropna(how="all").to_csv(f"{OUTPUT_DIR}/{t}.csv", index=False)
    print(f"Wrote {OUTPUT_DIR}/{t}.csv ({len(d)} rows)")

# consistency checks
raw_case_names = set(df.loc[df["case_type"] == "raw", "case_name"])
assert set(raw_tabs["raw_cases"]["raw_case"]) == raw_case_names, \
    f"raw_cases tab {sorted(raw_tabs['raw_cases']['raw_case'])} != raw cases in product_LCIs {sorted(raw_case_names)}"
for (case, stage), g in raw_tabs["raw_bom"].groupby(["raw_case", "stage"]):
    assert abs(g["percentage"].sum() - 100) < 0.01, f"raw_bom: {case}/{stage} sums to {g['percentage'].sum()}, not 100"
par = raw_tabs["raw_parameters"].pivot(index="raw_case", columns="parameter", values="value")
eol_cols = [c for c in par.columns if c.startswith("eol_share_")]
eol_sum = par[eol_cols].astype(float).sum(axis=1)
assert (eol_sum.round(6) == 100).all(), f"end-of-life shares must sum to 100: {eol_sum.to_dict()}"
print("OK - raw tabs consistent with product_LCIs; BOMs and end-of-life shares sum to 100")

# smoke test of the loader
import sys; sys.path.insert(0, ".")
from cases import load_raw_cases
loaded = load_raw_cases(OUTPUT_DIR)
print({k: (len(v.production_bom), len(v.production_chain)) for k, v in loaded.items()}, "(BOM items, production steps)")

Wrote ../../data/processed/raw_cases.csv (4 rows)
Wrote ../../data/processed/raw_bom.csv (14 rows)
Wrote ../../data/processed/raw_chain.csv (20 rows)
Wrote ../../data/processed/raw_parameters.csv (44 rows)
OK - raw tabs consistent with product_LCIs; BOMs and end-of-life shares sum to 100
{'raw_biopol': (3, 4), 'raw_timber': (1, 2), 'raw_cfw': (2, 2), 'raw_knit': (1, 3)} (BOM items, production steps)


### Carbon storage row for glulam

`benefits_constants.csv` (from 01) only covers RAW input materials. The glulam baseline stores biogenic carbon too, so it needs its own row: ecoinvent 'market for glued laminated timber, PUR-glue' | CH has carbon content non-fossil = 0.4469 kgC per kg dry mass, dry mass 399.46 and wet mass 439 kg/m3, so 0.4469 x 399.46 / 439 = 0.4067 kgC per kg wet. Rotation period: softwood (65 yr), same as 'Soft wood' in the sheet.

In [8]:
bp = f"{OUTPUT_DIR}/benefits_constants.csv"
dfb = pd.read_csv(bp)
if "glulam timber" not in dfb["material_name"].values:
    row = {c: None for c in dfb.columns}
    row.update(material_name="glulam timber", carbon_content_kgC_per_kgWet=round(0.446903840021865 * 399.46 / 439, 4),
               carbon_content_source="ecoinvent (glued laminated timber, PUR-glue | CH: carbon content non-fossil x dry mass / wet mass)",
               rotation_period_yr=65.0, rotation_period_source="Softwood CORRIM figure, same as Soft wood")
    dfb = pd.concat([dfb, pd.DataFrame([row])], ignore_index=True)
    dfb.to_csv(bp, index=False)
print(dfb[dfb["material_name"] == "glulam timber"].T)

                                                                             10
material_name                                                     glulam timber
carbon_content_kgC_per_kgWet                                             0.4067
carbon_content_source         ecoinvent (glued laminated timber, PUR-glue | ...
rotation_period_yr                                                         65.0
rotation_period_source                Softwood CORRIM figure, same as Soft wood


In [9]:
eol_path = f"{SOURCES_DIR}/baseline_eol_assumptions.csv"
eol = pd.read_csv(eol_path, comment="#")
share_sum = eol.groupby("activity_name")["share_pct"].sum()
assert (share_sum.round(6) == 100).all(), share_sum[share_sum.round(6) != 100]
missing = set(df.loc[df["case_type"].str.startswith("conventional"), "activity_name"]) - set(eol["activity_name"])
print("Baseline activities without an EoL row:", sorted(missing) or "none")
eol.to_csv(f"{OUTPUT_DIR}/baseline_eol.csv", index=False)
eol

Baseline activities without an EoL row: ['CFW beam']


,activity_name,kg_per_unit,route,share_pct,treatment_burden_name,lhv_MJ_per_kg,carbon_material,storage_yr
0,glulam timber,439,incinerated,60,NaN,17.7,glulam timber,50.0
1,glulam timber,439,recycled_open,30,NaN,NaN,NaN,NaN
2,glulam timber,439,landfilled,10,NaN,NaN,NaN,NaN
3,concrete,1,recycled_open,90,waste reinforced concrete recycling,NaN,NaN,NaN
4,concrete,1,landfilled,10,waste concrete landfilling,NaN,NaN,NaN
5,steel,1,recycled_open,90,waste reinforced concrete recycling,NaN,NaN,NaN
6,steel,1,landfilled,10,waste concrete landfilling,NaN,NaN,NaN
7,formwork,1,incinerated,100,NaN,14.2,NaN,NaN
8,adobe brick,1,recycled_closed,50,none,NaN,NaN,NaN
9,adobe brick,1,landfilled,50,waste inert landfilling,NaN,NaN,NaN
